# 🔐 FaceVault — Colab Demo

**Fast face recognition with anti-spoofing and vector database.**

This notebook demonstrates the full FaceVault workflow using a real dataset:

| User Code | Name | Images |
|-----------|------|--------|
| `OGH-00013` | Sumaya Kedir Jemel | 8 photos |
| `OGH-00044` | Meklit Ayele Woldeyes | 8 photos |
| `OGH-00238` | Afiya Kelifa Ahimed | 7 photos |

Test images: 10 unknown faces in `dataset/Test/`

---

## 1. Install & Clone

In [ ]:
# Clone the repository
!git clone https://github.com/abeldirectory252/face_vault.git
%cd face_vault

In [ ]:
# Install dependencies
!pip install -q insightface onnxruntime-gpu opencv-python-headless numpy scipy faiss-cpu Pillow

In [ ]:
# Verify
import face_vault
print(f"✅ FaceVault v{face_vault.__version__} loaded")

## 2. Check Dataset

The dataset ships with the repo under `dataset/`:

In [ ]:
import os
from pathlib import Path

dataset = Path("dataset")
print("📁 Dataset structure:\n")
for folder in sorted(dataset.iterdir()):
    if folder.is_dir():
        images = list(folder.glob("*.png")) + list(folder.glob("*.jpg"))
        print(f"  {folder.name}/  ({len(images)} images)")
        for img in sorted(images)[:3]:
            print(f"    └─ {img.name}")
        if len(images) > 3:
            print(f"    └─ ... and {len(images)-3} more")

## 3. Initialize FaceVault

In [ ]:
from face_vault import FaceVault, DetectMode
import cv2
import numpy as np
from IPython.display import display, Image as IPImage

def show_cv2(img, max_width=800):
    """Display a cv2 BGR image in Colab."""
    h, w = img.shape[:2]
    if w > max_width:
        scale = max_width / w
        img = cv2.resize(img, (max_width, int(h * scale)))
    _, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    display(IPImage(data=buf.tobytes()))

# Create the vault
vault = FaceVault(
    db_path="demo_faces.db",
    dataset_dir="dataset",
    ctx_id=0,               # 0 = GPU, -1 = CPU
    det_size=(640, 640),
    match_threshold=0.4,
    anti_spoof=True,
    spoof_block=False,      # warn only (for demo)
)
print("✅ FaceVault ready")

## 4. Register Identities

Register all three people from their dataset folders.
Each folder contains multiple photos of the same person.

In [ ]:
# ── Register Sumaya ─────────────────────────
r1 = vault.register(
    full_name="Sumaya Kedir Jemel",
    user_code="OGH-00013",
    reference_image="dataset/OGH-00013/",
)
print(f"[1] {r1.message}")

# ── Register Meklit ─────────────────────────
r2 = vault.register(
    full_name="Meklit Ayele Woldeyes",
    user_code="OGH-00044",
    reference_image="dataset/OGH-00044/",
)
print(f"[2] {r2.message}")

# ── Register Afiya ──────────────────────────
r3 = vault.register(
    full_name="Afiya Kelifa Ahimed",
    user_code="OGH-00238",
    reference_image="dataset/OGH-00238/",
)
print(f"[3] {r3.message}")

In [ ]:
# Verify registration
print("\n📊 Registered Identities:\n")
for ident in vault.list_identities():
    print(f"  [{ident.user_code}] {ident.name}")
    print(f"           vectors: {ident.num_vectors}")
    print(f"           ref:     {ident.reference_image_path}")
    print()

print(f"Stats: {vault.stats()}")

## 5. Test — DETECT_WITH_IDENTITY

Run each test image through the vault.
The system searches the **entire** database and tells us who it is.

In [ ]:
import glob

test_images = sorted(glob.glob("dataset/Test/unk*.png"))
print(f"🧪 Testing {len(test_images)} unknown images\n")
print(f"{'Image':<15} {'Matched':<9} {'Name':<25} {'Code':<12} {'Sim':>6} {'Time':>8}")
print("─" * 80)

for img_path in test_images:
    r = vault.identify(img_path, mode=DetectMode.DETECT_WITH_IDENTITY)
    name = r.name or "Unknown"
    code = r.user_code or "—"
    fname = os.path.basename(img_path)
    marker = "✅" if r.matched else "❌"
    print(f"{fname:<15} {marker:<9} {name:<25} {code:<12} {r.similarity:>6.4f} {r.elapsed_ms:>6.1f}ms")

## 6. Visualize — Stamped Overlay

Pick some test images and generate the annotated overlay with:
- Bounding box + name label
- VERIFIED / FAKE / UNCERTAIN stamp (top-left)
- Info panel (bottom-right)

In [ ]:
# Overlay only (no reference side-by-side)
for img_path in test_images[:3]:
    r = vault.identify(img_path, image_overlay=True)
    fname = os.path.basename(img_path)
    status = f"{r.name} [{r.user_code}]" if r.matched else "Unknown"
    print(f"\n🖼️ {fname} → {status}  (sim={r.similarity:.4f})")
    if r.image is not None:
        show_cv2(r.image)

## 7. Visualize — Side-by-Side Reference

When `reference_image=True`, the output shows the input probe
next to the matched person's registration photo.

A human judge can visually confirm: *is this really the same person?*

In [ ]:
# Side-by-side with reference image
for img_path in test_images[:3]:
    r = vault.identify(
        img_path,
        image_overlay=True,
        reference_image=True,
    )
    fname = os.path.basename(img_path)
    status = f"{r.name} [{r.user_code}]" if r.matched else "Unknown"
    print(f"\n👥 {fname} → {status}  (sim={r.similarity:.4f})")
    if r.image is not None:
        show_cv2(r.image, max_width=1200)

## 8. Test — DETECT_WITHOUT_IDENTITY

Fast 1:1 check: *"Is this unknown face actually Sumaya (OGH-00013)?"*

This skips the full FAISS scan and only compares against Sumaya's vectors.

In [ ]:
print("\n🔍 Checking each test image against OGH-00013 (Sumaya):\n")
print(f"{'Image':<15} {'Is Sumaya?':<12} {'Sim':>6} {'Time':>8}")
print("─" * 45)

for img_path in test_images:
    r = vault.identify(
        img_path,
        user_code="OGH-00013",
        mode=DetectMode.DETECT_WITHOUT_IDENTITY,
    )
    fname = os.path.basename(img_path)
    marker = "✅ YES" if r.matched else "❌ NO"
    print(f"{fname:<15} {marker:<12} {r.similarity:>6.4f} {r.elapsed_ms:>6.1f}ms")

In [ ]:
# Visualize one WITHOUT_IDENTITY check with overlay
r = vault.identify(
    test_images[0],
    user_code="OGH-00013",
    mode=DetectMode.DETECT_WITHOUT_IDENTITY,
    image_overlay=True,
)
fname = os.path.basename(test_images[0])
print(f"\n{fname} vs OGH-00013 (Sumaya): {'MATCH' if r.matched else 'NO MATCH'} (sim={r.similarity:.4f})")
if r.image is not None:
    show_cv2(r.image)

## 9. Cross-Check All Users

Test each unknown image against **all** registered users individually.

In [ ]:
users = [
    ("OGH-00013", "Sumaya"),
    ("OGH-00044", "Meklit"),
    ("OGH-00238", "Afiya"),
]

print(f"{'Image':<15}", end="")
for code, name in users:
    print(f"{name:>12}", end="")
print("    Best Match")
print("─" * 75)

for img_path in test_images:
    fname = os.path.basename(img_path)
    print(f"{fname:<15}", end="")

    best_code, best_sim = None, -1
    for code, name in users:
        r = vault.identify(
            img_path,
            user_code=code,
            mode=DetectMode.DETECT_WITHOUT_IDENTITY,
        )
        marker = "✅" if r.matched else "  "
        print(f"{marker}{r.similarity:>9.4f}", end="")
        if r.similarity > best_sim:
            best_sim = r.similarity
            best_code = code

    # Full identify for confirmation
    full = vault.identify(img_path)
    full_name = full.name or "Unknown"
    print(f"    → {full_name}")

## 10. Duplicate Detection

FaceVault rejects duplicate images (same SHA-256 hash).

In [ ]:
# Try registering the same images again — should be rejected
r_dup = vault.register(
    full_name="Sumaya Kedir Jemel",
    user_code="OGH-00013",
    reference_image="dataset/OGH-00013/",
)
print(f"Success: {r_dup.success}")
print(f"Message: {r_dup.message}")
print("\n✅ All 8 images were correctly detected as duplicates.")

## 11. Database Stats & Cleanup

In [ ]:
print("📊 Final Database Stats:\n")
for k, v in vault.stats().items():
    print(f"  {k}: {v}")

print("\n👤 Identities:")
for ident in vault.list_identities():
    print(f"  [{ident.user_code}] {ident.name} — {ident.num_vectors} vectors")

In [ ]:
vault.close()
print("✅ Done!")

---

## API Quick Reference

```python
from face_vault import FaceVault, DetectMode

vault = FaceVault("faces.db", dataset_dir="dataset")

# Register (folder of images)
vault.register(
    full_name="Afiya Kelifa Ahimed",
    user_code="OGH-00238",
    reference_image="dataset/OGH-00238/",
)

# Register (auto-discover from dataset/<user_code>/)
vault.register(full_name="Afiya Kelifa Ahimed", user_code="OGH-00238")

# Identify — who is this?
r = vault.identify("unknown.jpg")
print(r.matched, r.name, r.user_code)

# Identify — with annotated + reference side-by-side
r = vault.identify("unknown.jpg", image_overlay=True, reference_image=True)
cv2.imwrite("result.jpg", r.image)

# Fast check — is this person OGH-00238?
r = vault.identify("unknown.jpg", user_code="OGH-00238",
                   mode=DetectMode.DETECT_WITHOUT_IDENTITY)
print(r.matched, r.similarity)
```